# Présentation de la Base de Données Olist

Olist est une marketplace brésilienne qui met en relation des vendeurs indépendants et des acheteurs. Le dataset couvre **~100 000 commandes** passées entre 2016 et 2018 sur l'ensemble du territoire brésilien.

La base est organisée en **9 tables relationnelles** qui décrivent chaque étape du cycle de vie d'une commande.

---

### Schéma relationnel

```
olist_customers ──────────────────────────────────────────────┐
    customer_id                                               │
        │                                                     │
        ▼                                                     │
  olist_orders ──── olist_order_items ──── olist_products     │
    order_id              order_id              product_id    │
        │                    │                               │
        │                    └──────── olist_sellers         │
        │                                 seller_id          │
        ├──── olist_order_payments                           │
        │         order_id                                   │
        └──── olist_order_reviews                            │
                  order_id                                   │
                                                             │
olist_geolocation ◄── zip_code_prefix ───────────────────────┘
product_category_name_translation ◄── product_category_name
```

**Unité d'analyse :** `customer_unique_id` — identifiant pérenne du client, indépendant de ses adresses de livraison successives.

In [ ]:
import sys, os, subprocess, time
from pathlib import Path
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv

project_root = Path().resolve().parent
sys.path.append(str(project_root))
load_dotenv(project_root / ".env")

from src.data_loader import get_db_engine, get_merged_dataframe

PG_CONTAINER = "olist_postgres"

if subprocess.run(
    ["docker", "ps", "--filter", f"name={PG_CONTAINER}", "--format", "{{.Names}}"],
    capture_output=True, text=True
).stdout.strip():
    print(f"Conteneur '{PG_CONTAINER}' actif.")
else:
    subprocess.run(["docker", "start", PG_CONTAINER], check=True)
    time.sleep(3)
    print("Conteneur redemarre.")

engine = get_db_engine()

tables = [
    'olist_orders', 'olist_customers', 'olist_order_items',
    'olist_products', 'olist_sellers', 'olist_order_payments',
    'olist_order_reviews', 'olist_geolocation',
    'product_category_name_translation'
]
db_data = {t: pd.read_sql_table(t, engine) for t in tables}
print('Tables chargees :')
for t, df in db_data.items():
    print(f'  {t:<45} {len(df):>8,} lignes x {df.shape[1]} colonnes')

---
## Table 1 — `olist_orders`
**Rôle :** Table centrale du schéma. Chaque ligne représente **une commande**, avec son statut et ses horodatages clés.

| Colonne | Description |
|---------|-------------|
| `order_id` | Identifiant unique de la commande — clé primaire |
| `customer_id` | Identifiant de livraison du client (peut changer à chaque commande) |
| `order_status` | Statut : `delivered`, `shipped`, `canceled`, `invoiced`, `processing`… |
| `order_purchase_timestamp` | Date et heure de passage de la commande |
| `order_approved_at` | Date de validation du paiement |
| `order_delivered_carrier_date` | Date de remise au transporteur |
| `order_delivered_customer_date` | Date de livraison effective chez le client |
| `order_estimated_delivery_date` | Date de livraison estimée communiquée au client |

> **Usage projet :** `order_purchase_timestamp` → Recency. `order_delivered_customer_date - order_estimated_delivery_date` → `avg_delivery_delay`.

In [ ]:
df = db_data['olist_orders']
print(f"olist_orders : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Statuts : {df['order_status'].value_counts().to_dict()}")
display(df.head())

---
## Table 2 — `olist_customers`
**Rôle :** Référentiel client avec localisation géographique.

| Colonne | Description |
|---------|-------------|
| `customer_id` | Identifiant de livraison — clé étrangère vers `olist_orders` |
| `customer_unique_id` | **Identifiant pérenne du client** — unité d'analyse du projet |
| `customer_zip_code_prefix` | Code postal (5 chiffres) |
| `customer_city` | Ville du client |
| `customer_state` | État brésilien (2 lettres) : SP, RJ, MG… |

> **Usage projet :** Agrégation de toutes les commandes au niveau `customer_unique_id`. `customer_state` → `region_freight_score`.

In [ ]:
df = db_data['olist_customers']
print(f"olist_customers : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Clients uniques (customer_unique_id) : {df['customer_unique_id'].nunique():,}")
print(f"Etats couverts : {df['customer_state'].nunique()}")
display(df.head())

---
## Table 3 — `olist_order_items`
**Rôle :** Détail des articles de chaque commande. Une commande peut contenir plusieurs articles → plusieurs lignes par `order_id`.

| Colonne | Description |
|---------|-------------|
| `order_id` | Identifiant de la commande — clé étrangère |
| `order_item_id` | Numéro de l'article dans la commande (1, 2, 3…) |
| `product_id` | Identifiant du produit — clé étrangère vers `olist_products` |
| `seller_id` | Identifiant du vendeur — clé étrangère vers `olist_sellers` |
| `shipping_limit_date` | Date limite d'expédition imposée au vendeur |
| `price` | Prix unitaire de l'article (BRL) |
| `freight_value` | Frais de port de l'article (BRL) |

> **Usage projet :** `freight_value / price` → `avg_freight_ratio`. `sum(price + freight_value)` par client → `Monetary`.

In [ ]:
df = db_data['olist_order_items']
print(f"olist_order_items : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Commandes distinctes : {df['order_id'].nunique():,}")
print(f"Prix moyen : {df['price'].mean():.2f} BRL | Frais de port moyens : {df['freight_value'].mean():.2f} BRL")
display(df.head())

---
## Table 4 — `olist_products`
**Rôle :** Catalogue produits avec catégorie et caractéristiques physiques.

| Colonne | Description |
|---------|-------------|
| `product_id` | Identifiant unique du produit — clé primaire |
| `product_category_name` | Catégorie en portugais (ex: `cama_mesa_banho`) |
| `product_name_lenght` | Longueur du nom du produit (nb de caractères) |
| `product_description_lenght` | Longueur de la description (nb de caractères) |
| `product_photos_qty` | Nombre de photos du produit |
| `product_weight_g` | Poids en grammes |
| `product_length_cm` | Longueur en cm |
| `product_height_cm` | Hauteur en cm |
| `product_width_cm` | Largeur en cm |

> **Usage projet :** `product_category_name` (traduit) → `category_tier_encoded` et recommandations produits par cluster (Section 9b).

In [ ]:
df = db_data['olist_products']
print(f"olist_products : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Categories distinctes : {df['product_category_name'].nunique()}")
print(f"Top 5 categories :")
print(df['product_category_name'].value_counts().head().to_string())
display(df.head())

---
## Table 5 — `olist_sellers`
**Rôle :** Référentiel des vendeurs partenaires Olist avec leur localisation.

| Colonne | Description |
|---------|-------------|
| `seller_id` | Identifiant unique du vendeur — clé primaire |
| `seller_zip_code_prefix` | Code postal du vendeur |
| `seller_city` | Ville du vendeur |
| `seller_state` | État brésilien du vendeur |

> **Usage projet :** Analyse de la couverture géographique des vendeurs et impact sur les délais de livraison selon la distance vendeur-client.

In [ ]:
df = db_data['olist_sellers']
print(f"olist_sellers : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Etats couverts par les vendeurs : {sorted(df['seller_state'].unique())}")
display(df.head())

---
## Table 6 — `olist_order_payments`
**Rôle :** Détail des paiements par commande. Une commande peut avoir plusieurs lignes si le client combine plusieurs modes de paiement.

| Colonne | Description |
|---------|-------------|
| `order_id` | Identifiant de la commande — clé étrangère |
| `payment_sequential` | Numéro du paiement si combinaison de modes (1, 2…) |
| `payment_type` | Mode : `credit_card`, `boleto`, `voucher`, `debit_card` |
| `payment_installments` | Nombre de mensualités (1 = paiement comptant) |
| `payment_value` | Montant de ce paiement (BRL) |

> **Usage projet :** `mode(payment_type) == credit_card` → `payment_type_cc_flag`. `mean(payment_installments)` → `avg_installments`. `sum(payment_value)` → `Monetary`.

> **Boleto :** Équivalent brésilien du virement bancaire, très utilisé sans carte bancaire — proxy de revenu modeste.

In [ ]:
df = db_data['olist_order_payments']
print(f"olist_order_payments : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Repartition des modes de paiement :")
print(df['payment_type'].value_counts().to_string())
print(f"\nNb de versements moyen : {df['payment_installments'].mean():.1f}")
display(df.head())

---
## Table 7 — `olist_order_reviews`
**Rôle :** Avis clients laissés après livraison. Note de 1 (très insatisfait) à 5 (très satisfait).

| Colonne | Description |
|---------|-------------|
| `review_id` | Identifiant unique de l'avis |
| `order_id` | Identifiant de la commande évaluée — clé étrangère |
| `review_score` | Note de 1 à 5 étoiles |
| `review_comment_title` | Titre du commentaire (souvent vide) |
| `review_comment_message` | Texte du commentaire en portugais (souvent vide) |
| `review_creation_date` | Date d'envoi de la demande d'avis |
| `review_answer_timestamp` | Date de réponse du client |

> **Usage projet :** `mean(review_score)` par client → `avg_review_score`, indicateur de satisfaction et de fidélisabilité du segment.

In [ ]:
df = db_data['olist_order_reviews']
print(f"olist_order_reviews : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Repartition des notes :")
print(df['review_score'].value_counts().sort_index().to_string())
print(f"Note moyenne globale : {df['review_score'].mean():.2f} / 5")
display(df.head())

---
## Table 8 — `olist_geolocation`
**Rôle :** Coordonnées GPS associées à chaque code postal brésilien.

| Colonne | Description |
|---------|-------------|
| `geolocation_zip_code_prefix` | Code postal (5 chiffres) — clé de jointure |
| `geolocation_lat` | Latitude (degrés décimaux) |
| `geolocation_lng` | Longitude (degrés décimaux) |
| `geolocation_city` | Ville correspondante |
| `geolocation_state` | État brésilien |

> **Usage projet :** Visualisations cartographiques (choroplèthe des états) et construction de `region_freight_score`. Un même code postal peut avoir plusieurs entrées GPS — on prend la médiane.

In [ ]:
df = db_data['olist_geolocation']
print(f"olist_geolocation : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Codes postaux uniques : {df['geolocation_zip_code_prefix'].nunique():,}")
print(f"Latitude  : {df['geolocation_lat'].min():.2f} -> {df['geolocation_lat'].max():.2f}")
print(f"Longitude : {df['geolocation_lng'].min():.2f} -> {df['geolocation_lng'].max():.2f}")
display(df.head())

---
## Table 9 — `product_category_name_translation`
**Rôle :** Table de correspondance entre les noms de catégories en portugais et leur traduction en anglais.

| Colonne | Description |
|---------|-------------|
| `product_category_name` | Nom en portugais — clé de jointure vers `olist_products` |
| `product_category_name_english` | Traduction en anglais |

> **Exemple :** `cama_mesa_banho` → `bed_bath_table` | `esporte_lazer` → `sports_leisure`

> **Usage projet :** Jointure systématique pour afficher des labels lisibles dans l'EDA, le dashboard et les recommandations produits.

In [ ]:
df = db_data['product_category_name_translation']
print(f"product_category_name_translation : {len(df):,} lignes x {df.shape[1]} colonnes")
display(df.head(10))

---
## Table Maître — `df_master`
**Rôle :** DataFrame consolidé par jointure de toutes les tables. Table de travail principale pour le feature engineering.

**Grain :** 1 ligne = 1 article commandé, avec toutes les informations client, commande, produit et paiement associées.

```
olist_orders
    JOIN olist_customers      ON customer_id
    JOIN olist_order_items    ON order_id
    JOIN olist_products       ON product_id
    JOIN olist_sellers        ON seller_id
    JOIN olist_order_payments ON order_id
    LEFT JOIN olist_order_reviews ON order_id
    JOIN product_category_name_translation ON product_category_name
```

> Après jointure, on agrège au niveau `customer_unique_id` pour construire une **vue client unique** et calculer les features RFM.

In [ ]:
df_master = get_merged_dataframe(engine)
print(f"df_master : {len(df_master):,} lignes x {df_master.shape[1]} colonnes")
print(f"\nColonnes et taux de nulls :")
for col in df_master.columns:
    pct = df_master[col].isna().mean() * 100
    print(f"  {col:<45} {str(df_master[col].dtype):<12} {pct:>5.1f}% nulls")
print()
display(df_master.head())

---
## Récapitulatif

| Table | Lignes approx. | Colonnes | Rôle |
|-------|----------------|----------|------|
| `olist_orders` | ~99 000 | 8 | Table centrale — statuts et dates |
| `olist_customers` | ~99 000 | 5 | Référentiel client + localisation |
| `olist_order_items` | ~112 000 | 7 | Articles, prix, frais de port |
| `olist_products` | ~33 000 | 9 | Catalogue produits |
| `olist_sellers` | ~3 000 | 4 | Référentiel vendeurs |
| `olist_order_payments` | ~103 000 | 5 | Paiements et modes de règlement |
| `olist_order_reviews` | ~99 000 | 7 | Avis clients |
| `olist_geolocation` | ~1 000 000 | 5 | GPS par code postal |
| `product_category_name_translation` | 71 | 2 | Traduction catégories PT → EN |
| **`df_master`** | **~112 000** | **~30** | **Table de travail consolidée** |